# Customer value, spending trajectories and retention

Reference: **Two_Analytics_Projects_Complete_Guide.docx, Steps 1.3.1–1.3.5**. SQL: `sql/03_customers.sql`; reproducible runner: `scripts/run_customers.py`.

This notebook reads verified exports without taking a DBeaver database lock.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / 'exports/customers/hh_segments.csv').exists():
    ROOT = ROOT / 'outputs/grocery-marketing-effectiveness'
DATA = ROOT / 'exports/customers'
households = pd.read_csv(DATA / 'hh_segments.csv', dtype={'household_id': str})
segments = pd.read_csv(DATA / 'customer_segment_summary.csv')
trajectories = pd.read_csv(DATA / 'customer_trajectory_summary.csv')
cohorts = pd.read_csv(DATA / 'cohort_retention.csv', parse_dates=['cohort'])
income = pd.read_csv(DATA / 'income_spend.csv')
coverage = pd.read_csv(DATA / 'demographic_coverage_spend.csv')
checks = pd.read_csv(DATA / 'customer_checks.csv')
assert len(households) == 2467 and households.household_id.is_unique
assert checks.passed.astype(str).str.lower().eq('true').all()
assert abs(segments.value_share.sum() - 1) < 1e-6
print(f'{len(households):,} grocery households; {len(checks)} database checks passed')

## 1. Household metrics and segments

Snapshot: 31 December 2017. Lapsed status takes precedence. The guide uses grocery spend; two non-grocery-only households are outside this model. UNKNOWN-product purchases remain within the not-explicitly-non-grocery filter.

In [ ]:
display(households.head())
display(pd.read_csv(DATA / 'customer_cutoffs.csv'))
display(segments)
display(pd.read_csv(DATA / 'spend_concentration.csv'))
display(Image(filename=str(ROOT / 'charts/customer_segments.png')))

## 2. Spend trajectories

H1: source weeks 2–27; H2: 28–53. Growing means at least +20%; Declining means at least -20%. Partial week 1 is excluded from the comparison. New/Gone labels describe observed spending, not proven acquisition/churn. The observed decline is not a recoverable-revenue estimate.

In [ ]:
display(trajectories)
display(Image(filename=str(ROOT / 'charts/customer_trajectories.png')))

## 3. Monthly retention cohorts

Following the guide, this section uses all clean purchases (2,469 households), not the grocery-only segment population. First observed purchase is not proven acquisition. Elapsed inactive months are zero; unobservable future months are blank. Activity can resume after a missed month.

In [ ]:
display(cohorts[cohorts.months_since_first == 0][['cohort', 'cohort_size']])
assert cohorts.loc[cohorts.months_since_first == 0, 'cohort_size'].sum() == 2469
assert cohorts.retention.between(0, 1).all()
display(Image(filename=str(ROOT / 'charts/cohort_retention.png')))

## 4. Demographic attributes and spend

These are descriptive associations. The demographic subset spends more and does not represent the whole panel; small income bands are explicitly flagged. Missing attributes are not imputed.

In [ ]:
display(coverage)
display(income)
display(pd.read_csv(DATA / 'deal_driven_spend.csv'))
display(Image(filename=str(ROOT / 'charts/demographic_selection.png')))

## 5. Findings and validation

In [ ]:
display(checks)
display(Markdown((ROOT / 'docs/customers/findings.md').read_text(encoding='utf-8')))

## Next guide section

Module 4: promotion lift, followed by campaign incrementality. Customer descriptions above are not causal marketing estimates.